# Federated Ridge — Aggregating the Per-Study Vectors

**Coordinator notebook in the federated architecture.**

The four `Ridge_<STUDY>` notebooks each fit Ridge on one study alone and wrote
a coefficient vector to `vectors/`. This notebook is the coordinator: it
reads those vectors and combines them into a single federated model — **never
touching any study's subject-level data**, only the shared vectors.

It applies **three aggregation rules** so they can be compared:

1. **FedAvg-by-N** — weighted mean of the per-study vectors, weighted by each
   study's number of subjects. Large institutions count more.
2. **Simple mean** — unweighted mean. Every institution counts equally.
3. **Median (+ IQR)** — coordinate-wise median, with the inter-quartile range
   reported as a spread. Robust to one institution being an outlier.

## Three parts

- **Part 1 — Aggregate the shared vectors.** Read the CSVs, show the three
  combined vectors side by side. This is the transparent view: you can see
  exactly how each feature's per-study weights are combined into one number.
- **Part 2 — Honest federated evaluation.** A federated cross-validation:
  in each fold every study fits on its own training rows, the vectors are
  aggregated, and every study predicts its **own** held-out rows with the
  aggregated vector. This gives MSE / R² / power that are out-of-sample, not
  optimistic in-sample numbers. Still strictly per-study — no pooling.
- **Part 3 — Additive benefit.** Add the studies one at a time, **smallest N
  first**, and watch R² and power grow as the cohort grows. This is the
  federation thesis: more institutions → more power, without sharing data.

### A note on no pooling, including in scoring
Subject-level data never crosses a study boundary — not even to compute a
score. Each study evaluates the aggregated model on its **own** held-out rows
and reports only four summary numbers (residual sum of squares, Σy, Σy², n).
The coordinator combines those scalars into the global MSE / R². The numbers
are identical to scoring on pooled predictions, but no individual outcome is
ever shared.


## 1. Setup

In [ ]:
from __future__ import annotations
import sys, os, warnings
from pathlib import Path
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

REPO = Path.cwd()
if (REPO / "src").exists():
    sys.path.insert(0, str(REPO / "src"))
elif (REPO.parent / "src").exists():
    REPO = REPO.parent
    sys.path.insert(0, str(REPO / "src"))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import Ridge
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import KFold
from scipy.stats import f as fdist, ncf

import oadr_data as od

RNG_SEED = 42
RIDGE_ALPHA = 1.0
N_FOLDS = 5
ALPHA_STAT = 0.05
np.random.seed(RNG_SEED)
(REPO / "results").mkdir(exist_ok=True)
(REPO / "figures").mkdir(exist_ok=True)

PANEL_A_FEATS = ['MIAA', 'GAD65', 'IA2IC', 'ICA', 'ZNT8', '8-12', '13-17', '>18', 'Sex']
PANEL_B_FEATS = ['Sex', 'age_years', 'disease_duration_years', 'bmi', 'height_cm', 'weight_kg', 'GAD65', 'IA2IC', 'MIAA', 'ZNT8', 'ICA', 'received_active_treatment']
PANEL_A_STUDIES = ["SDY524", "SDY569", "SDY797", "SDY1737"]
PANEL_B_STUDIES = ["SDY524", "SDY569", "SDY1737"]
print("Repo root:", REPO)


## 2. Read the per-study vectors

Each `vectors/<STUDY>_ridge_panel<X>_vector.csv` holds one study's coefficient
vector, its intercept, and its sample size. We read them and stack them by
panel. The CSVs are the only thing that crosses the study boundary.


In [ ]:
def read_vectors(panel, studies, feats):
    """Return (coef_matrix [n_studies x n_feats], intercepts, sizes, study_names)."""
    coefs, intercepts, sizes, names = [], [], [], []
    for s in studies:
        p = REPO / "vectors" / f"{s}_ridge_panel{panel}_vector.csv"
        if not p.exists():
            print(f"  (skip {s}: no {p.name})")
            continue
        df = pd.read_csv(p).set_index("feature")
        intercepts.append(float(df.loc["__intercept__", "coefficient"]))
        coefs.append(df.loc[feats, "coefficient"].values.astype(float))
        sizes.append(int(df.loc[feats[0], "n_subjects"]))
        names.append(s)
    return np.array(coefs), np.array(intercepts), np.array(sizes), names

cA, iA, nA, namesA = read_vectors("A", PANEL_A_STUDIES, PANEL_A_FEATS)
cB, iB, nB, namesB = read_vectors("B", PANEL_B_STUDIES, PANEL_B_FEATS)
print("Panel A vectors:", namesA, "sizes", list(nA))
print("Panel B vectors:", namesB, "sizes", list(nB))


## 3. The three aggregation rules

Each rule takes the stack of per-study coefficient vectors and returns a single
federated vector.


In [ ]:
def fedavg_by_n(coefs, intercepts, sizes):
    return np.average(coefs, axis=0, weights=sizes), np.average(intercepts, weights=sizes)

def simple_mean(coefs, intercepts, sizes):
    return coefs.mean(axis=0), intercepts.mean()

def median_agg(coefs, intercepts, sizes):
    return np.median(coefs, axis=0), float(np.median(intercepts))

RULES = {"FedAvg_by_N": fedavg_by_n, "Simple_mean": simple_mean, "Median": median_agg}


## 4. Part 1 — the aggregated vectors, side by side

For each panel, the per-study coefficient for every feature, followed by the
three aggregations. The IQR column (Q3 − Q1 across studies) shows how much the
studies disagree on each feature — large IQR means the studies pull in
different directions (e.g. GAD65, whose sign flips across studies).


In [ ]:
def aggregate_table(coefs, intercepts, sizes, names, feats):
    tbl = pd.DataFrame(coefs.T, index=feats, columns=names)
    tbl["FedAvg_by_N"] = fedavg_by_n(coefs, intercepts, sizes)[0]
    tbl["Simple_mean"] = simple_mean(coefs, intercepts, sizes)[0]
    tbl["Median"]      = median_agg(coefs, intercepts, sizes)[0]
    tbl["IQR"] = np.percentile(coefs, 75, axis=0) - np.percentile(coefs, 25, axis=0)
    return tbl

print("=== Panel A aggregated coefficients ===")
tblA = aggregate_table(cA, iA, nA, namesA, PANEL_A_FEATS)
print(tblA.round(3).to_string())
tblA.to_csv("vectors/federated_ridge_panelA_aggregated.csv")
print()
print("=== Panel B aggregated coefficients ===")
tblB = aggregate_table(cB, iB, nB, namesB, PANEL_B_FEATS)
print(tblB.round(3).to_string())
tblB.to_csv("vectors/federated_ridge_panelB_aggregated.csv")


## 5. Part 2 — honest federated cross-validation

The aggregated vectors above were built from each study's **full** data, so
evaluating them on that same data would be optimistic. For honest numbers we
run a federated CV:

1. Split each study into K folds (each study splits its **own** rows).
2. In fold *k*: every study fits Ridge on its training rows (MinMax-scaled
   within that study), producing a vector. The vectors are aggregated by each
   rule.
3. Every study predicts its **own** held-out rows with the aggregated vector
   (scaling its held-out rows with its own training scaler).
4. Every study reports four summary scalars on its own rows (RSS, Σy, Σy², n);
   the coordinator combines them into the global MSE / R² / power.

We also record each study's **solo** prediction (its own vector on its own
held-out rows) so federation can be compared against going it alone.

Nothing is pooled: every fit, every scaler, every prediction uses one
study's rows only; rules combine vectors, scores combine summary scalars —
never subject-level data.


In [ ]:
def calc_power(n, k, f2, alpha=ALPHA_STAT):
    if n <= k + 1 or f2 <= 0:
        return float("nan")
    F_crit = fdist.ppf(1 - alpha, k, n - k - 1)
    return float(1 - ncf.cdf(F_crit, k, n - k - 1, f2 * n))

def f2_from_r2(r2):
    return r2 / (1 - r2) if 0 < r2 < 1 else 0.0

def study_summary(y, pred):
    """Four scalars a study can share without revealing any subject: RSS, Σy, Σy², n."""
    m = ~np.isnan(pred)
    yy, pp = y[m], pred[m]
    return {"rss": float(np.sum((yy - pp) ** 2)), "sum_y": float(np.sum(yy)),
            "sum_y2": float(np.sum(yy ** 2)), "n": int(m.sum())}

def score_from_summaries(summaries, k):
    """Combine per-study summary scalars into global MSE / R² / power. No subject data."""
    N   = sum(d["n"] for d in summaries)
    RSS = sum(d["rss"] for d in summaries)
    SY  = sum(d["sum_y"] for d in summaries)
    SY2 = sum(d["sum_y2"] for d in summaries)
    gmean = SY / N
    TSS = SY2 - N * gmean ** 2                 # Σ(y - global_mean)² from summary stats only
    mse = RSS / N
    r2 = 1 - RSS / TSS if TSS > 0 else float("nan")
    pwr = calc_power(N, k, f2_from_r2(r2))
    return mse, r2, N, pwr

def make_scaler(X_train):
    """Fit a within-study MinMaxScaler on X_train; returns a transform callable."""
    return MinMaxScaler().fit(X_train).transform


In [ ]:
def load_study(panel, study, feats):
    if panel == "A":
        a = od.load_panel_a(study)
        return a[feats].values.astype(float), a[od.PANEL_A_TARGET].values.astype(float)
    b = od.load_panel_b(study)
    for col in ("bmi", "height_cm", "weight_kg"):
        b[col] = b[col].fillna(b[col].median())
    bad = b["height_cm"] <= 0
    b.loc[bad, "height_cm"] = np.sqrt(b.loc[bad, "weight_kg"] / b.loc[bad, "bmi"]) * 100
    X, y, _ = od.panel_b_design_matrix(b)
    return X.reindex(columns=feats).values.astype(float), y.values.astype(float)

def federated_cv(panel, studies, feats, n_splits=N_FOLDS, seed=RNG_SEED):
    data = {s: load_study(panel, s, feats) for s in studies}     # each study loaded separately
    folds = {s: list(KFold(n_splits=min(n_splits, max(2, len(data[s][1]) // 2)),
                           shuffle=True, random_state=seed).split(data[s][0]))
             for s in studies}
    nf = min(len(folds[s]) for s in studies)
    # OOF arrays per study for each rule + solo
    oof = {r: {s: np.full(len(data[s][1]), np.nan) for s in studies} for r in list(RULES) + ["Solo"]}
    for k in range(nf):
        train_vecs = {}
        transformers = {}
        for s in studies:
            X, y = data[s]; tr, te = folds[s][k]
            tf = make_scaler(X[tr]); transformers[s] = tf
            m = Ridge(alpha=RIDGE_ALPHA).fit(tf(X[tr]), y[tr])
            train_vecs[s] = (m.coef_, m.intercept_, len(tr))
            # solo prediction: this study's own vector on its own held-out rows
            oof["Solo"][s][te] = m.predict(tf(X[te]))
        coefs = np.stack([train_vecs[s][0] for s in studies])
        ints = np.array([train_vecs[s][1] for s in studies])
        sizes = np.array([train_vecs[s][2] for s in studies])
        for rname, rule in RULES.items():
            ac, ai = rule(coefs, ints, sizes)
            for s in studies:
                X, y = data[s]; tr, te = folds[s][k]
                oof[rname][s][te] = transformers[s](X[te]) @ ac + ai
    # Each study reports only summary scalars on its OWN held-out rows; the
    # coordinator combines them. No subject-level data is concatenated.
    rows = []
    for rname in list(RULES) + ["Solo"]:
        summaries = [study_summary(data[s][1], oof[rname][s]) for s in studies]
        mse, r2, n, pwr = score_from_summaries(summaries, len(feats))
        rows.append({"panel": panel, "rule": rname, "N": n, "k": len(feats),
                     "MSE": mse, "R2": r2, "achieved_power": pwr})
    return pd.DataFrame(rows)

resA = federated_cv("A", PANEL_A_STUDIES, PANEL_A_FEATS)
resB = federated_cv("B", PANEL_B_STUDIES, PANEL_B_FEATS)
perf = pd.concat([resA, resB], ignore_index=True)
print(perf.to_string(index=False))
perf.to_csv("results/federated_ridge_performance.csv", index=False)


## 6. Federated vs solo

The `Solo` row is the average of each study predicting its own held-out rows
with its own vector — i.e. no federation. Compare the three aggregation rules
against it: if a rule's R² is higher (or its MSE lower) than `Solo`, sharing
vectors helped. If all rules and Solo are negative, neither solo nor federated
Ridge predicts C-peptide AUC from these features — an honest finding, not a bug.


In [ ]:
for panel in ("A", "B"):
    sub = perf[perf.panel == panel].set_index("rule")
    solo_r2 = sub.loc["Solo", "R2"]
    print(f"Panel {panel}: solo R²={solo_r2:+.3f}")
    for r in RULES:
        d = sub.loc[r, "R2"] - solo_r2
        print(f"   {r:14s} R²={sub.loc[r,'R2']:+.3f}  (vs solo: {d:+.3f})   MSE={sub.loc[r,'MSE']:.3f}  power={sub.loc[r,'achieved_power']:.3f}")
    print()


## 7. Graphics — coefficients and performance

Two figures per panel:

- **Aggregation comparison** — each feature's four per-study coefficients
  (small dots) plus the three aggregated values (coloured markers). Where the
  dots straddle zero (e.g. GAD65), the studies disagree on direction and the
  aggregations land near zero — the cancellation made visible.
- **Performance bars** — R² and achieved power for the three rules and the
  solo baseline.


In [ ]:
def plot_aggregation(tbl, feats, names, panel):
    fig, ax = plt.subplots(figsize=(8, 0.45 * len(feats) + 1.5))
    y = np.arange(len(feats))[::-1]
    for s in names:                                   # per-study dots
        ax.scatter(tbl.loc[feats, s], y, s=22, color="0.6", zorder=2)
    marks = {"FedAvg_by_N": ("o", "#1f77b4"), "Simple_mean": ("s", "#ff7f0e"),
             "Median": ("D", "#2ca02c")}
    for rule, (mk, col) in marks.items():
        ax.scatter(tbl.loc[feats, rule], y, marker=mk, s=70, color=col,
                   edgecolor="white", linewidth=0.6, zorder=3, label=rule)
    ax.axvline(0, color="k", lw=0.6)
    ax.set_yticks(y); ax.set_yticklabels(feats)
    ax.set_xlabel("Ridge coefficient (MinMax-scaled)")
    ax.set_title(f"Aggregation of per-study vectors — Panel {panel}", fontweight="bold")
    ax.legend(loc="best", fontsize=8); ax.grid(axis="x", alpha=0.3)
    fig.tight_layout()
    fig.savefig(f"figures/federated_ridge_coef_panel{panel}.pdf", dpi=300)
    fig.savefig(f"figures/federated_ridge_coef_panel{panel}.png", dpi=220)
    plt.show()

plot_aggregation(tblA, PANEL_A_FEATS, namesA, "A")
plot_aggregation(tblB, PANEL_B_FEATS, namesB, "B")


In [ ]:
def plot_performance(panel):
    sub = perf[perf.panel == panel].set_index("rule")
    order = ["Solo", "FedAvg_by_N", "Simple_mean", "Median"]
    fig, axes = plt.subplots(1, 2, figsize=(11, 4), constrained_layout=True)
    colors = ["0.6", "#1f77b4", "#ff7f0e", "#2ca02c"]
    axes[0].bar(order, [sub.loc[r, "R2"] for r in order], color=colors)
    axes[0].axhline(0, color="k", lw=0.6); axes[0].set_ylabel("R² (federated CV)")
    axes[0].set_title(f"R² — Panel {panel}", fontweight="bold")
    axes[0].tick_params(axis="x", rotation=20)
    pw = [sub.loc[r, "achieved_power"] for r in order]
    pw = [0 if np.isnan(v) else v for v in pw]
    axes[1].bar(order, pw, color=colors)
    axes[1].axhline(0.80, color="r", ls="--", lw=1, label="80% power")
    axes[1].set_ylim(0, 1.0); axes[1].set_ylabel("Achieved power")
    axes[1].set_title(f"Power — Panel {panel}", fontweight="bold")
    axes[1].tick_params(axis="x", rotation=20); axes[1].legend(fontsize=8)
    fig.savefig(f"figures/federated_ridge_performance_panel{panel}.pdf", dpi=300)
    fig.savefig(f"figures/federated_ridge_performance_panel{panel}.png", dpi=220)
    plt.show()

plot_performance("A")
plot_performance("B")


## 8. Part 3 — additive benefit (cohort growth, smallest → largest)

The federation thesis: **adding institutions raises statistical power without
sharing data.** We add the studies one at a time, **smallest N first**, and
re-run the federated CV at each step.

- Panel A order: SDY569 (10) → +SDY1737 (16) → +SDY797 (49) → +SDY524 (75)
- Panel B order: SDY569 (10) → +SDY1737 (16) → +SDY524 (72)

For each cohort size we plot R² and achieved power, one line per aggregation
rule. A rising power curve is the additive benefit made visible.


In [ ]:
GROWTH_ORDER = {
    "A": ["SDY569", "SDY1737", "SDY797", "SDY524"],     # smallest N first
    "B": ["SDY569", "SDY1737", "SDY524"],
}

def cohort_growth(panel, feats):
    order = GROWTH_ORDER[panel]
    records = []
    for i in range(1, len(order) + 1):
        subset = order[:i]
        res = federated_cv(panel, subset, feats).set_index("rule")
        cohort_n = int(res.loc["FedAvg_by_N", "N"])
        for rule in list(RULES) + ["Solo"]:
            records.append({"panel": panel, "n_studies": i, "cohort_N": cohort_n,
                            "added": subset[-1], "rule": rule,
                            "R2": res.loc[rule, "R2"],
                            "achieved_power": res.loc[rule, "achieved_power"]})
    return pd.DataFrame(records)

growthA = cohort_growth("A", PANEL_A_FEATS)
growthB = cohort_growth("B", PANEL_B_FEATS)
growth = pd.concat([growthA, growthB], ignore_index=True)
growth.to_csv("results/federated_ridge_cohort_growth.csv", index=False)
print(growth.to_string(index=False))


In [ ]:
def plot_growth(panel, growth_df):
    order = GROWTH_ORDER[panel]
    labels = [f"{i}: +{order[i-1]}" for i in range(1, len(order) + 1)]
    colors = {"FedAvg_by_N": "#1f77b4", "Simple_mean": "#ff7f0e",
              "Median": "#2ca02c", "Solo": "0.6"}
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.4), constrained_layout=True)
    for rule, col in colors.items():
        d = growth_df[growth_df.rule == rule].sort_values("n_studies")
        axes[0].plot(d.n_studies, d.R2, marker="o", color=col, label=rule)
        pw = d.achieved_power.fillna(0.0)
        axes[1].plot(d.n_studies, pw, marker="o", color=col, label=rule)
    axes[0].axhline(0, color="k", lw=0.6); axes[0].set_ylabel("R² (federated CV)")
    axes[0].set_xlabel("cohort (studies added, smallest first)")
    axes[0].set_xticks(range(1, len(order) + 1)); axes[0].set_xticklabels(labels, rotation=20, ha="right")
    axes[0].set_title(f"R² grows with cohort — Panel {panel}", fontweight="bold")
    axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)
    axes[1].axhline(0.80, color="r", ls="--", lw=1, label="80% power")
    axes[1].set_ylim(0, 1.0); axes[1].set_ylabel("Achieved power")
    axes[1].set_xlabel("cohort (studies added, smallest first)")
    axes[1].set_xticks(range(1, len(order) + 1)); axes[1].set_xticklabels(labels, rotation=20, ha="right")
    axes[1].set_title(f"Power grows with cohort — Panel {panel}", fontweight="bold")
    axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)
    fig.savefig(f"figures/federated_ridge_cohort_growth_panel{panel}.pdf", dpi=300)
    fig.savefig(f"figures/federated_ridge_cohort_growth_panel{panel}.png", dpi=220)
    plt.show()

plot_growth("A", growthA)
plot_growth("B", growthB)


## 9. Outputs

CSVs:
- `vectors/federated_ridge_panel{A,B}_aggregated.csv` — per-study + three
  aggregated vectors, with IQR.
- `results/federated_ridge_performance.csv` — federated-CV MSE / R² / power
  per rule + solo, both panels.
- `results/federated_ridge_cohort_growth.csv` — R² and power at each cohort
  size (smallest → largest), per rule, both panels.

Figures (`figures/`):
- `federated_ridge_coef_panel{A,B}` — aggregation comparison.
- `federated_ridge_performance_panel{A,B}` — R² and power bars.
- `federated_ridge_cohort_growth_panel{A,B}` — additive-benefit curves.

This is the Ridge arm of the federation. The same pattern (per-study notebooks →
this coordinator) will be repeated for LASSO and, with a different aggregation,
random forest.
